# 05 - Treinar GMM (Regime Detection)

GMM para identificar regimes de mercado.
- Se regime_prob < threshold: BLOCK trade

**IMPORTANTE**: Após treinar, mapear Cluster ID -> Block/Pass baseado em performance histórica.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.mixture import GaussianMixture

plt.style.use('dark_background')
DATA_DIR = Path('../data')
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(exist_ok=True)

In [ ]:
# Carregar features
with open(DATA_DIR / 'features.pkl', 'rb') as f:
    data = pickle.load(f)

X_regime = data['X_regime']
y = data['y']

print(f'X_regime shape: {X_regime.shape}')

## 1. Encontrar Número Ótimo de Clusters

In [ ]:
# Testar diferentes números de clusters
n_components_range = range(2, 8)
bic_scores = []
aic_scores = []

for n in n_components_range:
    gmm = GaussianMixture(n_components=n, random_state=42, n_init=5)
    gmm.fit(X_regime)
    bic_scores.append(gmm.bic(X_regime))
    aic_scores.append(gmm.aic(X_regime))

# Plotar
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_components_range, bic_scores, 'o-', label='BIC')
ax.plot(n_components_range, aic_scores, 's-', label='AIC')
ax.set_xlabel('Número de Clusters')
ax.set_ylabel('Score')
ax.set_title('BIC/AIC vs Número de Clusters')
ax.legend()
plt.tight_layout()
plt.show()

# Escolher menor BIC
optimal_n = n_components_range[np.argmin(bic_scores)]
print(f'\nNúmero ótimo de clusters: {optimal_n}')

## 2. Treinar GMM Final

In [ ]:
# Treinar GMM com n ótimo
gmm = GaussianMixture(
    n_components=optimal_n, 
    random_state=42, 
    n_init=10,
    covariance_type='full'
)
gmm.fit(X_regime)

# Predições
clusters = gmm.predict(X_regime)
probs = gmm.predict_proba(X_regime)

print(f'Clusters encontrados: {np.unique(clusters)}')
print(f'Distribuição: {np.bincount(clusters)}')

## 3. Mapear Cluster -> Performance

In [ ]:
# Calcular win rate por cluster
df_clusters = pd.DataFrame({
    'cluster': clusters,
    'y': y
})

cluster_stats = df_clusters.groupby('cluster')['y'].agg(['mean', 'count'])
cluster_stats.columns = ['win_rate', 'count']
cluster_stats['win_rate'] = cluster_stats['win_rate'] * 100

print('=== Performance por Cluster ===')
print(cluster_stats)

# Identificar clusters ruins (win_rate < 45%)
bad_clusters = cluster_stats[cluster_stats['win_rate'] < 45].index.tolist()
print(f'\nClusters para BLOQUEAR: {bad_clusters}')

In [ ]:
# Visualizar
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['red' if c in bad_clusters else 'lime' for c in cluster_stats.index]
cluster_stats['win_rate'].plot(kind='bar', ax=ax, color=colors)
ax.axhline(50, color='white', linestyle='--', label='50% baseline')
ax.set_xlabel('Cluster')
ax.set_ylabel('Win Rate (%)')
ax.set_title('Win Rate por Regime/Cluster')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Salvar Modelo

In [ ]:
# Salvar GMM e mapeamento
gmm_data = {
    'model': gmm,
    'bad_clusters': bad_clusters,
    'cluster_stats': cluster_stats,
}

with open(MODEL_DIR / 'gmm_regime.pkl', 'wb') as f:
    pickle.dump(gmm_data, f)

print(f'GMM salvo em: {MODEL_DIR / "gmm_regime.pkl"}')

## 5. Próximos Passos

1. **06_train_xgb.ipynb**: Treinar XGBoost com X_signal
2. Combinar GMM + XGBoost no pipeline final